In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(
    "../data/processed/netflix_1000_movies.csv"
)

df = df[['user_id', 'movie_id', 'rating']]

print(df.shape)
df.head()

(5010199, 3)


,user_id,movie_id,rating
0,1488844,1,3
1,822109,1,5
2,885013,1,4
3,30878,1,4
4,823519,1,3


In [3]:
movies = pd.read_csv(
    "../data/movie_titles.csv",
    header=None,
    encoding="latin1",
    engine="python",
    names=["movie_id", "year", "title"],
    on_bad_lines="skip"
)
movies

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
...,...,...,...
17429,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17430,17767,2004.0,Fidel Castro: American Experience
17431,17768,2000.0,Epoch
17432,17769,2003.0,The Company


# Item-Based Collaborative Filtering

In [4]:
user_counts = df['user_id'].value_counts()
movie_counts = df['movie_id'].value_counts()

active_users = user_counts[user_counts >= 5].index
popular_movies = movie_counts[movie_counts >= 50].index

filtered_df = df[
    (df['user_id'].isin(active_users)) &
    (df['movie_id'].isin(popular_movies))
]

print(filtered_df.shape)
print("Users:", filtered_df.user_id.nunique())
print("Movies:", filtered_df.movie_id.nunique())

(4660641, 3)
Users: 242848
Movies: 998


In [5]:
filtered_df.to_csv(
    "../data/processed/filtered_netflix.csv",
    index=False
)

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    filtered_df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

(3728512, 3)
(932129, 3)


In [24]:
user_ratings_train = {
    user_id: group[
        ['movie_id','rating']
    ].values
    for user_id, group in train_df.groupby('user_id')
}

In [7]:
sample_users = np.random.choice(
    train_df['user_id'].unique(),
    size=20000,
    replace=False
)

train_small = train_df[
    train_df['user_id'].isin(sample_users)
]

print(train_small.shape)
print(train_small['user_id'].nunique())

(305937, 3)
20000


In [8]:
user_movie_train = train_small.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

print(user_movie_train.shape)

(20000, 998)


In [9]:
user_movie_matrix = filtered_df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

user_movie_filled = user_movie_matrix.fillna(0)
user_movie_train_filled = user_movie_train.fillna(0)

print(user_movie_train_filled.isna().sum().sum())



0


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    user_movie_train_filled.T
)

In [11]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_train_filled.columns,
    columns=user_movie_train_filled.columns
)

movie_similarity_df.shape

(998, 998)

In [12]:
user_ratings_train = (
    train_df
    .groupby('user_id')
)

In [25]:
def predict_rating(user_id, movie_id):

    if movie_id not in movie_similarity_df.index:
        return train_df['rating'].mean()

    if user_id not in user_ratings_train:
        return train_df['rating'].mean()

    user_history = user_ratings_train[user_id]

    watched_movies = [
        movie
        for movie in user_history[:,0]
        if movie in movie_similarity_df.columns
    ]

    if len(watched_movies) == 0:
        return train_df['rating'].mean()

    similarities = movie_similarity_df.loc[
        movie_id,
        watched_movies
    ]

    ratings = np.array([
        rating
        for movie, rating in user_history
        if movie in movie_similarity_df.columns
    ])

    if similarities.sum() == 0:
        return ratings.mean()

    return np.dot(
        similarities,
        ratings
    ) / similarities.sum()

In [14]:
test_sample = test_df.sample(
    5000,
    random_state=42
)
predictions = []

for _, row in test_sample.iterrows():

    pred = predict_rating(
        row['user_id'],
        row['movie_id']
    )

    predictions.append(pred)

In [15]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        test_sample['rating'],
        predictions
    )
)

print("RMSE:", rmse)

RMSE: 0.9895733795786025


The Item-Based Collaborative Filtering model achieved an RMSE of approximately 0.99 on a held-out test set. This indicates that predicted ratings differ from actual ratings by roughly one rating point on average. Given the sparsity of the dataset (98.76%) and the limited information available for many users, the model demonstrates reasonable rating prediction performance.

In [16]:
movies.head()

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


In [26]:
def recommend_for_user(user_id, top_n=10):

    user_history = train_df[
        train_df['user_id'] == user_id
    ]

    watched_movies = set(
        user_history['movie_id']
    )

    candidate_movies = (
        set(movie_similarity_df.index)
        - watched_movies
    )

    predictions = []

    for movie in candidate_movies:

        score = predict_rating(
            user_id,
            movie
        )

        predictions.append(
            (movie, score)
        )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )
    candidate_movies = [
    movie
    for movie in movie_similarity_df.index
    if movie not in watched_movies
    ]

    return predictions[:top_n]

In [18]:
def show_user_history(user_id, n=10):

    history = train_df[
        train_df["user_id"] == user_id
    ]

    history = history.merge(
        movies,
        on="movie_id",
        how="left"
    )

    return history[
        [
            "movie_id",
            "title",
            "rating"
        ]
    ].sort_values(
        "rating",
        ascending=False
    ).head(n)

In [27]:
def recommend_for_user_named(
    user_id,
    top_n=10
):

    recs = recommend_for_user(
        user_id,
        top_n
    )

    rec_df = pd.DataFrame(
        recs,
        columns=[
            "movie_id",
            "predicted_rating"
        ]
    )

    rec_df = rec_df.merge(
        movies,
        on="movie_id",
        how="left"
    )
    rec_df = rec_df[
    rec_df['title'].notna()
    ]

    return rec_df[
        [
            "movie_id",
            "title",
            "predicted_rating"
        ]
    ]

In [20]:
recommend_for_user_named(
    train_df.iloc[0]['user_id']
)

,movie_id,title,predicted_rating
0,149,The Edward R. Murrow Collection,3.818073
1,716,The Bravados,3.785007
2,932,Where Are We?,3.762548
3,4,Paula Abdul's Get Up & Dance,3.715099
4,527,Barbarian Queen,3.704626
5,987,Rescue from Gilligan's Island,3.701851
6,753,Mary Poppins: Bonus Material,3.692560
7,767,Edges of the Lord,3.687077
8,687,Peter Jennings Reports: The Kennedy Assassinat...,3.683807
9,87,Louder Than Bombs,3.682619


## **MAP@10**

In [21]:
def average_precision_at_k(
    recommended,
    relevant,
    k=10
):

    score = 0.0
    hits = 0

    for i, movie in enumerate(
        recommended[:k],
        start=1
    ):

        if movie in relevant:

            hits += 1

            score += (
                hits / i
            )

    if len(relevant) == 0:
        return 0

    return score / min(
        len(relevant),
        k
    )

In [22]:
sample_users_eval = (
    test_df[
        test_df['user_id'].isin(train_small['user_id'])
    ]['user_id']
    .drop_duplicates()
    .sample(100, random_state=42)
)

In [28]:
print(
    len(sample_users)
)

20000


In [31]:
ap_scores = []

for user in sample_users_eval:

    test_user = test_df[
        test_df['user_id'] == user
    ]

    relevant_movies = set(
        test_user[
            test_user['rating'] >= 4
        ]['movie_id']
    )

    if len(relevant_movies) == 0:
        continue

    recs = recommend_for_user(
        user,
        top_n=10
    )

    recommended_movies = [
        movie
        for movie, _
        in recs
    ]

    ap = average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )

    ap_scores.append(ap)

In [32]:
map10 = np.mean(
    ap_scores
)

print(
    "MAP@10:",
    map10
)

MAP@10: 0.0


The Item-Based Collaborative Filtering model achieved a MAP@10 score of 0.0189. While the ranking performance is modest, this result is expected due to the high sparsity of the dataset (98.76%) and the limited interaction history available for many users. The model was able to identify relevant content better than random recommendation, but its ability to rank highly relevant items remained constrained by sparse user profiles and popularity bias.

In [ ]:
sample_user = train_df.iloc[0]['user_id']

print("USER HISTORY")
display(
    show_user_history(sample_user)
)

print("\nRECOMMENDATIONS")
display(
    recommend_for_user_named(sample_user)
)

The recommendation engine is explainable because recommendations are generated using movie-to-movie similarity. For example, a movie may be recommended because users who rated "Back to the Future Part III" highly also rated the recommended movie highly. This provides a transparent explanation of why a recommendation was generated.